In [ ]:
import lightning as L
import timm
import torch
from pytorch_metric_learning import miners
import torchvision.transforms as T
import cv2
from torch.utils.data import DataLoader
import os
import random

In [1]:
class CarsEmbedder(L.LightningModule):
    def __init__(self, margin=1.0, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name='efficientnet_b3', pretrained=True, num_classes=0)

        self.loss = torch.nn.TripletMarginLoss(margin=margin)
        self.miner = miners.TripletMarginMiner(margin=margin, type_of_triplets="semihard")

        self.validation_step_outputs = []
        self.validation_step_labels = []

    def forward(self, imgs):
        return self.model(imgs)

    def training_step(self, batch):
        imgs, labels = batch

        embeds = self(imgs)

        anchors_indices, positives_indices, negatives_indices = self.miner(embeds, labels)
        loss = self.loss(embeds[anchors_indices], embeds[positives_indices], embeds[negatives_indices])
        self.log('train_loss', loss)
        return loss

    @staticmethod
    def precision_k_for_one(main_embed, main_label, other_embeds, labels, k=5):
        res = 0
        distances = []
        for j, other_embed in enumerate(other_embeds):
            distance = torch.nn.functional.cosine_similarity(main_embed, other_embed)
            distances.append((j, distance))
        nearest = sorted(distances, key=lambda x: x[1], reverse=True)[1:k+1]
        res += sum(1 for idx, dist in nearest if labels[idx] == main_label) / k
        return res

    @staticmethod
    def recall_k(embeds, labels, k=5):
        res = 0
        for i, embed in enumerate(embeds):
            distances = []
            main_label = labels[i]
            main_label_count = max(1, sum(1 for label in labels if label == main_label) - 1)
            for j, other_embeds in enumerate(embeds):
                distance = torch.nn.functional.cosine_similarity(embed, other_embeds)
                distances.append((j, distance))
            nearest = sorted(distances, key=lambda x: x[1], reverse=True)[1:k+1]
            res += sum(1 for idx, dist in nearest if labels[idx] == main_label) / main_label_count
        return res / len(embeds)

    @staticmethod
    def m_ap(embeds, labels):
        res = 0
        for i, embed in enumerate(embeds):
            distances = []
            main_label = labels[i]
            for j, other_embed in enumerate(embeds):
                distance = torch.nn.functional.cosine_similarity(embed, other_embed)
                distances.append((j, distance))
            ranked = sorted(distances, key=lambda x: x[1], reverse=True)[1:]

            relevants_count = sum(1 for label in labels if label == main_label)
            res += sum(CarsEmbedder.precision_k_for_one(embed, main_label, embeds, labels, k) for k in range(1, len(ranked)+1) if labels[ranked[k-1][0]] == main_label) / relevants_count
        return res / len(embeds)



    def validation_step(self, batch):
        imgs, labels = batch

        embeds = self(imgs)

        anchors_indices, positives_indices, negatives_indices = self.miner(embeds, labels)
        loss = self.loss(embeds[anchors_indices], embeds[positives_indices], embeds[negatives_indices])
        self.log("val_loss", loss, on_step=False, on_epoch=True)

        self.validation_step_outputs.append(embeds.detach().cpu())
        self.validation_step_labels.append(labels.detach().cpu())

    def on_validation_epoch_end(self):

        embeds = torch.cat(self.validation_step_outputs, dim=0)
        labels = torch.cat(self.validation_step_labels, dim=0)

        prec_k = 0
        for embed, label in zip(embeds, labels):
            prec_k += self.precision_k_for_one(embed, label, embeds, labels, k=5)
        prec_k /= len(embeds)
        rec_k = self.recall_k(embeds, labels)
        m_ap = self.m_ap(embeds, labels)
        self.log(f"val_precision@k", prec_k)
        self.log(f"val_recall@k", rec_k)
        self.log("val_mAP", m_ap)

        self.validation_step_labels.clear()
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

IndentationError: expected an indented block after function definition on line 22 (2323487646.py, line 25)

In [ ]:
ADD_PATH = '.\\cars_train'
BATCH_SIZE = 32

In [ ]:
class CarsDataset(torch.utils.data.Dataset):
    def __init__(self, data, transforms=None):
        self.data = data
        if transforms is not None:
            self.transforms = transforms
        else:
            self.transforms = T.Compose([T.ToPILImage(), T.Resize((224, 224)), T.ToTensor(), ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename, cl_id = self.data[idx]
        image = cv2.imread(os.path.join(ADD_PATH, filename))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transforms:
            image = self.transforms(image)
        return image, cl_id

In [2]:
import scipy.io
mat = scipy.io.loadmat('./devkit/cars_train_annos.mat')
fname_to_class = {fname:cl-1 for fname, cl in zip([i[0] for i in mat['annotations'][0]['fname']],
                                                [i[0][0] for i in mat['annotations'][0]['class']])}
cars_meta = scipy.io.loadmat('./devkit/cars_meta.mat')
id_to_car = {idx: car[0] for idx, car in enumerate(cars_meta['class_names'][0])}
print(fname_to_class)

{np.str_('00001.jpg'): np.uint8(13), np.str_('00002.jpg'): np.uint8(2), np.str_('00003.jpg'): np.uint8(90), np.str_('00004.jpg'): np.uint8(133), np.str_('00005.jpg'): np.uint8(105), np.str_('00006.jpg'): np.uint8(122), np.str_('00007.jpg'): np.uint8(88), np.str_('00008.jpg'): np.uint8(95), np.str_('00009.jpg'): np.uint8(166), np.str_('00010.jpg'): np.uint8(57), np.str_('00011.jpg'): np.uint8(48), np.str_('00012.jpg'): np.uint8(185), np.str_('00013.jpg'): np.uint8(134), np.str_('00014.jpg'): np.uint8(84), np.str_('00015.jpg'): np.uint8(192), np.str_('00016.jpg'): np.uint8(171), np.str_('00017.jpg'): np.uint8(13), np.str_('00018.jpg'): np.uint8(72), np.str_('00019.jpg'): np.uint8(191), np.str_('00020.jpg'): np.uint8(56), np.str_('00021.jpg'): np.uint8(78), np.str_('00022.jpg'): np.uint8(35), np.str_('00023.jpg'): np.uint8(119), np.str_('00024.jpg'): np.uint8(169), np.str_('00025.jpg'): np.uint8(193), np.str_('00026.jpg'): np.uint8(133), np.str_('00027.jpg'): np.uint8(183), np.str_('00028

In [ ]:
items = list(fname_to_class.items())
random.shuffle(items)
train_items = items[:int(len(items) * 0.8)]
val_items = items[int(len(items) * 0.8):]

train_dataset = CarsDataset(train_items)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataset = CarsDataset(val_items)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)